In [16]:
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from neuralhydrology.datasetzoo import get_dataset, camelsus
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.shm import SHM
from neuralhydrology.nh_run import start_run
from neuralhydrology.utils.config import Config
from neuralhydrology.training.basetrainer import BaseTrainer

### Train the LSTM
To start, let's train an lstm for a single basin. If you're curious this is for the Narraguagus River with flow as measured at Cherryfield, Maine. I chose this for consistency with `examples/05-Inspecting-LSTMs`

In [ ]:
config_file = Path("1_basin.yml")
# by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
# if torch.cuda.is_available() or torch.backends.mps.is_available():
#     start_run(config_file=config_file)

# # fall back to CPU-only mode
# else:
#     start_run(config_file=config_file, gpu=-1)

2026-02-13 11:39:22,959: Logging to /Users/danielmckenzie/Documents/Active_Research/Ziyu/neuralhydrology-dcfe/examples/09-Data-Assimilation/runs/test_run_1302_113922/output.log initialized.
2026-02-13 11:39:22,960: ### Folder structure created at /Users/danielmckenzie/Documents/Active_Research/Ziyu/neuralhydrology-dcfe/examples/09-Data-Assimilation/runs/test_run_1302_113922
2026-02-13 11:39:22,960: ### Run configurations for test_run
2026-02-13 11:39:22,961: experiment_name: test_run
2026-02-13 11:39:22,961: train_basin_file: 1_basin.txt
2026-02-13 11:39:22,961: validation_basin_file: 1_basin.txt
2026-02-13 11:39:22,962: test_basin_file: 1_basin.txt
2026-02-13 11:39:22,962: train_start_date: 1999-10-01 00:00:00
2026-02-13 11:39:22,963: train_end_date: 2008-09-30 00:00:00
2026-02-13 11:39:22,963: validation_start_date: 1980-10-01 00:00:00
2026-02-13 11:39:22,963: validation_end_date: 1989-09-30 00:00:00
2026-02-13 11:39:22,964: test_start_date: 1989-10-01 00:00:00
2026-02-13 11:39:22,96

KeyboardInterrupt: 

### Load up trained model
The results of training (model weights, metadata, and optimizer-related data) are saved in `runs`. Let's load them up.

In [4]:
run_dir = Path('runs/test_run_1202_165626')  # this value comes from the output of the above command
!ls $run_dir/model_epoch* | tail -n 3

runs/test_run_1202_165626/model_epoch028.pt
runs/test_run_1202_165626/model_epoch029.pt
runs/test_run_1202_165626/model_epoch030.pt


### Load up trained model
Let's create a new instance of the neural network and then load the trained weights into it.

In [5]:
cudalstm_config = Config(config_file)

# create a new model instance with random weights
cuda_lstm = CudaLSTM(cfg=cudalstm_config)

# load the trained weights into the new model. 
model_path = run_dir / 'model_epoch030.pt'
model_weights = torch.load(str(model_path), map_location='cpu')  # load the weights from the file, creating the weight tensors on CPU
cuda_lstm.load_state_dict(model_weights)  # set the new model's weights to the values loaded from file
cuda_lstm

CudaLSTM(
  (embedding_net): InputLayer(
    (statics_embedding): Identity()
    (dynamics_embeddings): ModuleList(
      (0): Identity()
    )
  )
  (lstm): LSTM(5, 20)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Regression(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=1, bias=True)
    )
  )
)

### Configuring SHM
Now, let's initialize an SHM model.

In [ ]:
shm_config_file = Path("shm_config.yml")
shm_config = Config(shm_config_file)
shm = SHM(cfg=shm_config)

SHM()

### Fetch the data
Lets instantiate a dataloader containing the data we want

In [ ]:
trainer = BaseTrainer(config_file)
loader = trainer.loader

AttributeError: 'PosixPath' object has no attribute 'allow_subsequent_nan_losses'

In [ ]:
## ToDo:
# 1. Fetch decent parameters, or make them up.
# 1.5 initialize the data loader. 
# 2. use shm.timestep in a for loop.
# 3. How do I connect this to forcings?